In [2]:
# packages
import pandas as pd
import numpy as np
import matplotlib as mpl
#import chartgazing as cg
from bs4 import BeautifulSoup
import requests
import json
import re

In [40]:
# import pokemon names and numbers
url = "https://www.pokellector.com/Base-Set-Expansion/"

res = requests.get(url)
soup = BeautifulSoup(res.content, "html.parser")

pokemon = soup.select('.plaque')

pokemon_names = []

for entry in pokemon:
    try:
        pokemon_names.append(entry.get_text())
    except Exception as e:
        print(f"An error occurred: {e}")
        
print(pokemon_names[0:5])

pokemon_df = pd.DataFrame(pokemon_names, columns=['pokemon'])

pokemon_df["number"] = pokemon_df["pokemon"].str.extract(r"#\s*([^\s-]+)", expand=False)
pokemon_df["name"] = pokemon_df["pokemon"].str.extract(r"\-(.*)", expand=False).str.strip().str.lower()

pokemon_df

['#1 - Alakazam', '#2 - Blastoise', '#3 - Chansey', '#4 - Charizard', '#5 - Clefairy']


,pokemon,number,name
0,#1 - Alakazam,1,alakazam
1,#2 - Blastoise,2,blastoise
2,#3 - Chansey,3,chansey
3,#4 - Charizard,4,charizard
4,#5 - Clefairy,5,clefairy
...,...,...,...
97,#98 - Fire Energy,98,fire energy
98,#99 - Grass Energy,99,grass energy
99,#100 - Lightning Energy,100,lightning energy
100,#101 - Psychic Energy,101,psychic energy


In [54]:
# test import of price data from pricecharting.com
def poke_pull(name, number):
    """Pull a Pokemon cards value history from pricecharting.com"""
    url = f"https://www.pricecharting.com/game/pokemon-base-set/{name}-1st-edition-{number}#completed-auctions-used"

    html = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}).text

    chart_data = json.loads(re.search(r"VGPC\.chart_data\s*=\s*(\{.*?\});", html, re.S).group(1))

    df = pd.DataFrame(
        [(grade, ts, price) for grade, points in chart_data.items() for ts, price, in points],
        columns =["grade", "date", "price"]
    )

    df["date"] = pd.to_datetime(df["date"], unit="ms")
    df["price"] = df["price"] / 100
    df["name"] = name
    df["number"] = number

    return df

print(poke_pull(pokemon_df["name"][0], pokemon_df["number"][0]))

       grade                date   price      name number
0    boxonly 2020-12-01 07:00:00    0.00  alakazam      1
1    boxonly 2021-01-01 07:00:00    0.00  alakazam      1
2    boxonly 2021-02-01 07:00:00    0.00  alakazam      1
3    boxonly 2021-03-01 07:00:00    0.00  alakazam      1
4    boxonly 2021-04-01 06:00:00    0.00  alakazam      1
..       ...                 ...     ...       ...    ...
421     used 2026-06-01 06:00:00  550.00  alakazam      1
422     used 2026-07-01 06:00:00  677.18  alakazam      1
423     used 2026-08-01 06:00:00  648.00  alakazam      1
424     used 2026-09-01 06:00:00  597.32  alakazam      1
425     used 2026-10-01 06:00:00  770.00  alakazam      1

[426 rows x 5 columns]


In [ ]:
# https://www.pricecharting.com/game/pokemon-base-set/charizard-1st-edition-4#completed-auctions-used
# #1 - Alakazam

pokemon_all = pd.DataFrame(columns=['grade', 'date', 'price', 'name', 'number'])

for name, number in zip(pokemon_df["name"], pokemon_df["number"]):
    try:
        data = poke_pull(name, number)
        pokemon_all = pd.concat([pokemon_all, data], ignore_index=True)
        print(f"Successfull added #{number} - {name}")
    except Exception as e:
        print(f"Error occured: {e}")


Successfull added #1 - alakazam
Successfull added #2 - blastoise
Successfull added #3 - chansey
Successfull added #4 - charizard
Successfull added #5 - clefairy
Successfull added #6 - gyarados
Successfull added #7 - hitmonchan
Successfull added #8 - machamp
Successfull added #9 - magneton
Successfull added #10 - mewtwo
Successfull added #11 - nidoking
Successfull added #12 - ninetales
Successfull added #13 - poliwrath
Successfull added #14 - raichu
Successfull added #15 - venusaur
Successfull added #16 - zapdos
Successfull added #17 - beedrill
Successfull added #18 - dragonair
Successfull added #19 - dugtrio
Successfull added #20 - electabuzz
Successfull added #21 - electrode
Successfull added #22 - pidgeotto
Successfull added #23 - arcanine
Successfull added #24 - charmeleon
Successfull added #25 - dewgong
Successfull added #26 - dratini
Successfull added #27 - farfetch'd
Successfull added #28 - growlithe
Successfull added #29 - haunter
Successfull added #30 - ivysaur
Successfull adde

In [62]:
print(pokemon_all.dtypes)
print(pokemon_all.info())

grade     object
date      object
price     object
name      object
number    object
dtype: object
<class 'pandas.DataFrame'>
RangeIndex: 14340 entries, 0 to 14339
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   grade   14340 non-null  object
 1   date    14340 non-null  object
 2   price   14340 non-null  object
 3   name    14340 non-null  object
 4   number  14340 non-null  object
dtypes: object(5)
memory usage: 560.3+ KB
None
